# Shopora — serve your watch try-on model

Loads **SDXL inpainting** plus the LoRA trained in `train_watch_lora.ipynb` and exposes an
`/enhance` endpoint. Shopora sends the 3D AR try-on result and a mask of the wrist area; the model
repaints only that area at low strength, so the watch keeps its exact design but gains realistic
skin contact, shadows and lighting.

**Runtime:** GPU (T4 is enough). Colab sessions stop after a while, so this is for testing and
demos. For real customers, run the same code on a GPU host (see `README.md`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install diffusers==0.31.0 transformers==4.46.3 accelerate==1.1.1 peft==0.13.2 safetensors \
    fastapi==0.115.5 uvicorn==0.32.1 python-multipart
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared

## 1. Configure

`LORA_DIR` is the training notebook's `OUTPUT_DIR`. Leave it empty to try the base model first.
Copy the printed **API key** into Shopora's `.env.local` as `TRYON_ENHANCE_KEY`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import secrets

LORA_DIR = '/content/drive/MyDrive/shopora-tryon/loras/wrist-watches'
LORA_SCALE = 0.8
TRIGGER = ''   # set to the training TRIGGER for a product-mode LoRA, e.g. 'shpwtch'

API_KEY = secrets.token_urlsafe(24)
print('TRYON_ENHANCE_KEY =', API_KEY)

## 2. Load the model

In [ ]:
import os
import torch
from diffusers import AutoencoderKL, AutoPipelineForInpainting

vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=torch.float16)
pipe = AutoPipelineForInpainting.from_pretrained(
    'diffusers/stable-diffusion-xl-1.0-inpainting-0.1', vae=vae, torch_dtype=torch.float16, variant='fp16'
).to('cuda')

if LORA_DIR and os.path.exists(f'{LORA_DIR}/pytorch_lora_weights.safetensors'):
    pipe.load_lora_weights(LORA_DIR, weight_name='pytorch_lora_weights.safetensors')
    pipe.fuse_lora(lora_scale=LORA_SCALE)
    print('LoRA loaded from', LORA_DIR)
else:
    print('No LoRA found — using the base inpainting model')

## 3. The enhance step

The wrist is usually a small part of the photo, so the masked area is cropped to a square,
processed at 1024 px and pasted back with a feathered edge. Everything outside the mask is
untouched.

In [ ]:
import threading
from PIL import Image, ImageFilter, ImageOps

NEGATIVE_PROMPT = (
    'cgi, 3d render, plastic, cartoon, illustration, blurry, deformed hand, extra fingers, '
    'floating watch, misaligned strap, watermark, text'
)
gpu_lock = threading.Lock()

def build_prompt(product: str = '') -> str:
    watch = f'{TRIGGER} watch' if TRIGGER else (product or 'watch')
    return (
        f"a realistic photo of a person's wrist wearing a {watch}, strap resting on the skin, "
        'natural contact shadows, matching scene lighting, sharp focus, high detail'
    )

def enhance(image: Image.Image, mask: Image.Image, prompt: str, strength: float = 0.45,
            steps: int = 40, guidance: float = 6.0, seed: int = -1, padding: float = 0.6) -> Image.Image:
    image = ImageOps.exif_transpose(image).convert('RGB')
    mask = mask.convert('L').resize(image.size)
    bbox = mask.point(lambda v: 255 if v > 127 else 0).getbbox()
    if bbox is None:
        return image

    # Square crop around the masked area, with context so lighting and skin tone carry over.
    x0, y0, x1, y1 = bbox
    side = int(max(x1 - x0, y1 - y0) * (1 + padding))
    side = max(256, min(side, image.width, image.height))
    cx, cy = (x0 + x1) // 2, (y0 + y1) // 2
    left = min(max(0, cx - side // 2), image.width - side)
    top = min(max(0, cy - side // 2), image.height - side)
    box = (left, top, left + side, top + side)

    crop = image.crop(box).resize((1024, 1024), Image.LANCZOS)
    crop_mask = mask.crop(box).resize((1024, 1024), Image.LANCZOS)
    generator = torch.Generator('cuda').manual_seed(seed) if seed >= 0 else None
    with gpu_lock, torch.inference_mode():
        out = pipe(
            prompt=prompt, negative_prompt=NEGATIVE_PROMPT, image=crop, mask_image=crop_mask,
            strength=strength, num_inference_steps=steps, guidance_scale=guidance, generator=generator,
        ).images[0]

    out = out.resize((side, side), Image.LANCZOS)
    feather = mask.crop(box).filter(ImageFilter.GaussianBlur(max(2, side // 60)))
    result = image.copy()
    result.paste(out, box[:2], feather)
    return result

## 4. Try it on a photo

Download a result from Shopora's try-on debug page (`/prototype/photo-try-on-debug`, see
`README.md`) and upload the `*-composite.png` and `*-mask.png` files here. Compare strengths:
lower keeps the watch closer to the 3D render, higher adds more realism but can change details.

In [ ]:
from google.colab import files
from IPython.display import display

uploaded = files.upload()   # pick the composite and the mask
composite = Image.open(next(name for name in uploaded if 'mask' not in name))
mask = Image.open(next(name for name in uploaded if 'mask' in name))

for strength in (0.3, 0.45, 0.6):
    result = enhance(composite, mask, build_prompt(), strength=strength, seed=0)
    print('strength', strength)
    display(result.resize((result.width // 2, result.height // 2)))

## 5. Serve it to Shopora

Starts the API and a free Cloudflare tunnel. Put the printed URL in Shopora's `.env.local` as
`TRYON_ENHANCE_URL` and restart `npm run dev`. The URL changes every time this cell runs.

In [ ]:
import io, re, subprocess
import uvicorn
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.concurrency import run_in_threadpool
from fastapi.responses import Response

app = FastAPI()

@app.get('/health')
def health():
    return {'ok': True, 'lora': bool(LORA_DIR and os.path.exists(f'{LORA_DIR}/pytorch_lora_weights.safetensors'))}

@app.post('/enhance')
async def enhance_endpoint(
    image: UploadFile = File(...),
    mask: UploadFile = File(...),
    product: str = Form(''),
    strength: float = Form(0.45),
    seed: int = Form(-1),
    x_api_key: str = Header(None),
):
    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail='Invalid API key')
    source = Image.open(io.BytesIO(await image.read()))
    region = Image.open(io.BytesIO(await mask.read()))
    strength = min(max(strength, 0.15), 0.8)
    result = await run_in_threadpool(enhance, source, region, build_prompt(product[:80]), strength, 40, 6.0, seed)
    buffer = io.BytesIO()
    result.save(buffer, format='JPEG', quality=92)
    return Response(content=buffer.getvalue(), media_type='image/jpeg')

server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning'))
threading.Thread(target=server.run, daemon=True).start()

tunnel = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stderr=subprocess.PIPE, text=True,
)
for line in tunnel.stderr:
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        print('TRYON_ENHANCE_URL =', match.group(0))
        print('TRYON_ENHANCE_KEY =', API_KEY)
        break